trying using uproot and awkward instead of pyroot

optimising this version for NOTS.

#### 1. Part 1 builds the 2D 2-particle correlation in both the standard and WTA frames. Clusters the jets according to the two definitions, transforms things to the jet frame, makes the signal distribution, the background distribution, and then the final normalised yield.

BEFORE RUNNING:
1. check input (f = cell) and output (def saveOutput) locations
2. check multiplicity bins (in final analysis cell)


In [54]:
import ROOT
import math
import fastjet
import os
import ctypes
import gc
import awkward as ak
import uproot
import numpy as np
import vector
vector.register_awkward()

In [4]:
# 1. Force ROOT into headless batch mode - i.e. turn off pop up graphics
ROOT.gROOT.SetBatch(True)

In [3]:
jet_radius = 0.8 #max radius accepted by fastjet is 1000. radius value in prl paper is 0.8 in lab frame.

wta_def = fastjet.JetDefinition(fastjet.antikt_algorithm, jet_radius, fastjet.WTA_pt_scheme) #winner take all definition
std_def = fastjet.JetDefinition(fastjet.antikt_algorithm, jet_radius) #standard E scheme definition

In [49]:
# INPUT DATA IS OPENED HERE

#f = uproot.Open("/storage/hpc/work/wl33/ampt_xiao/3mb/nch60_pt500") # maybe use with open as file:

tree = uproot.open("/Users/rohanjagadeesan/Desktop/Code/li_lab/WinnerTakeAll/data/pp_parton_cascade_batch0_0.root:trackTree")

data = tree.arrays(["genJetPt", "genJetEta", "genDau_pt", "genDau_eta", "genDau_phi", "genDau_chg"]) #opening only the relevant columns

#del tree

tree.show()



name                 | typename                 | interpretation                
---------------------+--------------------------+-------------------------------
par_pdgid            | std::vector<int32_t>     | AsJagged(AsDtype('>i4'), he...
par_px               | std::vector<float>       | AsJagged(AsDtype('>f4'), he...
par_py               | std::vector<float>       | AsJagged(AsDtype('>f4'), he...
par_pz               | std::vector<float>       | AsJagged(AsDtype('>f4'), he...
par_e                | std::vector<float>       | AsJagged(AsDtype('>f4'), he...
par_x                | std::vector<float>       | AsJagged(AsDtype('>f4'), he...
par_y                | std::vector<float>       | AsJagged(AsDtype('>f4'), he...
par_z                | std::vector<float>       | AsJagged(AsDtype('>f4'), he...
par_t                | std::vector<float>       | AsJagged(AsDtype('>f4'), he...
par_color1           | std::vector<int32_t>     | AsJagged(AsDtype('>i4'), he...
par_color2           | std::

In [53]:
print(tree.keys())

['par_pdgid', 'par_px', 'par_py', 'par_pz', 'par_e', 'par_x', 'par_y', 'par_z', 'par_t', 'par_color1', 'par_color2', 'par_pdgid_after_zpc', 'par_px_after_zpc', 'par_py_after_zpc', 'par_pz_after_zpc', 'par_e_after_zpc', 'par_x_after_zpc', 'par_y_after_zpc', 'par_z_after_zpc', 'par_t_after_zpc', 'par_color1_after_zpc', 'par_color2_after_zpc', 'total_collisions', 'px', 'py', 'pz', 'm', 'pid', 'chg', 'genJetEta', 'genJetPt', 'genJetPhi', 'genJetChargedMultiplicity', 'Zgs', 'Rgs', 'ZgTgBs', 'SDJetMass', 'genDau_chg', 'genDau_pid', 'genDau_pt', 'genDau_eta', 'genDau_phi']


In [50]:
pars = tree.arrays("par_pdgid")
pars.show()


[{par_pdgid: [1, 2, 21, 21, 21, 21, 21, ..., 2203, -1, 1, -1, 2, 3, 2]},
 {par_pdgid: [-1, 21, -2, 21, 21, 21, 21, ..., -2, -1, -2, 1, -1, 1, -2]},
 {par_pdgid: [21, 21, 21, 21, 21, 21, 21, ..., 2, 2, -2, 1, 2, -3, -2]},
 {par_pdgid: [2, 21, 21, 21, -2, 21, 21, ..., 2101, 1, 1, -3, -1, 1]},
 {par_pdgid: [2, 21, 21, 21, 21, 21, 21, ..., 2, 1, -2, -3, 3, -1, -1]},
 {par_pdgid: [21, -4, 21, 21, 21, -3, -1, ..., 2, -1, 2, 2101, 3, 1, -1]},
 {par_pdgid: [21, 2, 21, 21, 21, 21, 21, ..., 2101, 1, 3, 3, 2, -1, -1]},
 {par_pdgid: [21, 21, 21, 3, 2, 21, 3, 21, ..., 1, -3, -3, -3, 1, 2, -1]},
 {par_pdgid: [-1, 21, 3, 21, 21, 2, 21, 21, ..., 2, -3, -1, 1, 2, 1, 2]},
 {par_pdgid: [-3, 21, 21, -3, 21, 2, 21, ..., 2203, 1, -2, 2101, 2, -1]},
 ...,
 {par_pdgid: [2, 4, 3, 21, -3, 1, 21, 21, ..., -1, -1, 2101, 2, 2, -2, 2]},
 {par_pdgid: [21, 21, 21, 2, 21, 21, -4, ..., 2101, -1, 3, -3, 2103, 1]},
 {par_pdgid: [21, -4, 21, 21, 21, 21, 21, ..., 2, 2203, 1, 4, 1, 3, -2]},
 {par_pdgid: [-1, 21, 21, 2, 21, 

In [ ]:
# lab frame jet cuts:

jetEtaCut = 1.6 #from PRL paper page 2
jetPtCut = 550  #from prl paper. This is the STANDARD pt, not wta pt.

data = data[(data.genJetPt > jetPtCut) & (abs(data.genJetEta) < jetEtaCut)]
data = data[ak.num(data.genJetPt) > 0]

data.show()


[{genJetPt: [617], genJetEta: [-0.8], genDau_pt: [[...]], genDau_eta: ..., ...},
 {genJetPt: [819], genJetEta: [0.00482], genDau_pt: [[...]], ...},
 {genJetPt: [564], genJetEta: [-0.342], genDau_pt: [[...]], ...},
 {genJetPt: [557], genJetEta: [0.416], genDau_pt: [[...]], ...},
 {genJetPt: [555], genJetEta: [-0.915], genDau_pt: [[...]], ...},
 {genJetPt: [670], genJetEta: [1.15], genDau_pt: [[...]], genDau_eta: ..., ...},
 {genJetPt: [749], genJetEta: [-0.176], genDau_pt: [[...]], ...},
 {genJetPt: [563], genJetEta: [1.35], genDau_pt: [[...]], genDau_eta: ..., ...},
 {genJetPt: [900], genJetEta: [-0.922], genDau_pt: [[...]], ...},
 {genJetPt: [580], genJetEta: [0.58], genDau_pt: [[...]], genDau_eta: ..., ...},
 ...,
 {genJetPt: [1.58e+03], genJetEta: [-0.324], genDau_pt: [[...]], ...},
 {genJetPt: [718], genJetEta: [0.142], genDau_pt: [[...]], ...},
 {genJetPt: [1.07e+03], genJetEta: [-0.215], genDau_pt: [[...]], ...},
 {genJetPt: [740], genJetEta: [-0.882], genDau_pt: [[...]], ...},
 

In [57]:
#flattening to the relevant jets

# Flatten the event/jet axes so that axis 0 is simply a list of all passing jets
jet_dau_pt = ak.flatten(data.genDau_pt, axis=1)
jet_dau_eta = ak.flatten(data.genDau_eta, axis=1)
jet_dau_phi = ak.flatten(data.genDau_phi, axis=1)
jet_dau_chg = ak.flatten(data.genDau_chg, axis=1)

energies = jet_dau_pt * np.cosh(jet_dau_eta) # calculates total momentum, approximately equal to particle energy.

# 1. Build a jagged array of 4-vectors for all particles inside all jets
particles = ak.zip({
    "pt": jet_dau_pt,
    "eta": jet_dau_eta,
    "phi": jet_dau_phi,
    "E": energies 
}, with_name="Momentum4D")

particles.show()

[[{pt: 1.41, eta: -1.08, phi: 2.71, E: 2.31}, ..., {pt: 28.1, eta: ..., ...}],
 [{pt: 1.88, eta: 0.576, phi: 0.406, E: 2.21}, ..., {pt: 110, eta: ..., ...}],
 [{pt: 0.0653, eta: -1.82, phi: 1.32, E: 0.206}, ..., {pt: 110, eta: ..., ...}],
 [{pt: 2.74, eta: 0.411, phi: 2.57, E: 2.98}, ..., {pt: 17.6, eta: 0.186, ...}],
 [{pt: 0.561, eta: 0.19, phi: 5.94, E: 0.572}, ..., {pt: 22.6, eta: ..., ...}],
 [{pt: 0.0955, eta: -1.69, phi: 2.4, E: 0.267}, ..., {pt: 36.7, eta: ..., ...}],
 [{pt: 1.19, eta: 0.507, phi: 0.392, E: 1.35}, ..., {pt: 17.7, eta: 1.02, ...}],
 [{pt: 0.292, eta: -2.01, phi: 1.72, E: 1.1}, ..., {pt: 33.7, eta: -1.71, ...}],
 [{pt: 1.66, eta: 0.263, phi: 0.662, E: 1.72}, ..., {pt: 24.4, eta: ..., ...}],
 [{pt: 0.455, eta: 0.853, phi: 6.08, E: 0.63}, ..., {pt: 37.3, eta: 1.28, ...}],
 ...,
 [{pt: 0.859, eta: -0.66, phi: 0.217, E: 1.05}, ..., {pt: 12.3, eta: ..., ...}],
 [{pt: 1.22, eta: 0.204, phi: 1.54, E: 1.24}, ..., {pt: 49.1, eta: ..., ...}],
 [{pt: 0.919, eta: -0.256, phi

In [58]:
# applying lab frame particle cuts

particlePtCut = 0.3 #from prl paper page 2
particleEtaCut = 2.4 #from prl paper page 2

# 2. Vectorized particle selection (replaces your particlePass function)
particle_mask = (particles.pt > particlePtCut) & (abs(particles.eta) < particleEtaCut) & (jet_dau_chg != 0) #charged particles, meeting pt and eta cuts
particles = particles[particle_mask] #now only valid particles left

particles.show()

[[{pt: 1.41, eta: -1.08, phi: 2.71, E: 2.31}, ..., {pt: 16.2, eta: ..., ...}],
 [{pt: 1.88, eta: 0.576, phi: 0.406, E: 2.21}, ..., {pt: 110, eta: ..., ...}],
 [{pt: 0.648, eta: -0.224, phi: 0.252, E: 0.664}, {...}, ..., {pt: 110, ...}],
 [{pt: 0.341, eta: 1.6, phi: 3.07, E: 0.875}, ..., {pt: 17.6, eta: 0.186, ...}],
 [{pt: 0.726, eta: 0.154, phi: 0.347, E: 0.735}, {...}, ..., {pt: 36.2, ...}],
 [{pt: 0.355, eta: -1.61, phi: 2.07, E: 0.923}, ..., {pt: 34.4, eta: ..., ...}],
 [{pt: 1.19, eta: 0.507, phi: 0.392, E: 1.35}, ..., {pt: 50.4, eta: 1.02, ...}],
 [{pt: 0.665, eta: -1.32, phi: 3.18, E: 1.34}, ..., {pt: 33.7, eta: ..., ...}],
 [{pt: 1.66, eta: 0.263, phi: 0.662, E: 1.72}, ..., {pt: 24.4, eta: ..., ...}],
 [{pt: 1.13, eta: 1.41, phi: 6.24, E: 2.44}, ..., {pt: 12.1, eta: 1.28, ...}],
 ...,
 [{pt: 0.859, eta: -0.66, phi: 0.217, E: 1.05}, ..., {pt: 12.3, eta: ..., ...}],
 [{pt: 0.457, eta: 0.311, phi: 0.368, E: 0.479}, {...}, ..., {pt: 49.1, ...}],
 [{pt: 4.68, eta: -0.155, phi: 1.12,

In [164]:
wta_clustered = fastjet.ClusterSequence(particles, wta_def)
std_clustered = fastjet.ClusterSequence(particles, std_def)


wta_jets = fastjet.sorted_by_pt(wta_clustered.inclusive_jets())
std_jets = fastjet.sorted_by_pt(std_clustered.inclusive_jets())


#wta_jets = wta_jets[:, -1] #slicing to be only the leading jet in the recombined lists. Other items in the list will only have a couple of soft particles
#std_jets = std_jets[:, -1]



In [158]:
wta_jets.show()

[[{px: -2.09, py: 0.952, pz: -2.99, E: 3.77}, {px: -400, py: -139, ...}],
 [{px: 568, py: -43.5, pz: 5.68, E: 570}],
 [{px: 207, py: 320, pz: -170, E: 417}],
 [{px: -2.26, py: -0.217, pz: 2.95, E: 3.73}, {px: -325, py: -96.1, ...}],
 [{px: 5.63, py: -0.733, pz: 1.08, E: 5.78}, {px: 340, py: 11.8, ...}],
 [{px: -0.171, py: 0.312, pz: -0.852, E: 0.923}, {...}, {px: -56.2, ...}],
 [{px: 1.97, py: -2.07, pz: 3.52, E: 4.53}, {px: 405, py: 8.45, ...}],
 [{px: -0.664, py: -0.0239, pz: -1.16, E: 1.34}, {px: -336, py: 270, ...}],
 [{px: 112, py: 48, pz: 45.1, E: 130}, {px: 405, py: -25.3, pz: ..., ...}],
 [{px: 5.12, py: -0.529, pz: 10.9, E: 12}, {px: 262, py: -316, ...}],
 ...,
 [{px: 0.752, py: -0.108, pz: 0.711, E: 1.04}, {px: 517, py: 92.9, ...}],
 [{px: 569, py: 593, pz: -143, E: 835}],
 [{px: 265, py: 430, pz: -534, E: 735}],
 [{px: 0.42, py: 0.328, pz: -1.65, E: 1.74}, ..., {px: 291, py: 211, ...}],
 [{px: -0.334, py: -0.454, pz: 1.89, E: 1.98}, ..., {px: -172, py: -308, ...}],
 [{px: 0.

In [159]:
print(wta_jets)

[[{px: -2.09, py: 0.952, pz: -2.99, E: 3.77}, {px: -400, ...}], ..., [...]]


In [156]:
wta_over550 = wta_clustered.inclusive_jets(550)
print(wta_over550)
print("next")
print(wta_over550[ak.num(wta_over550)>0])
print("next")
print(wta_over550)

[[], [{px: 568, py: -43.5, pz: 5.68, E: 570}], [], ..., [{px: -224, ...}], []]
next
[[{px: 568, py: -43.5, pz: 5.68, E: 570}], ..., [{px: -224, py: 602, ...}]]
next
[[], [{px: 568, py: -43.5, pz: 5.68, E: 570}], [], ..., [{px: -224, ...}], []]


In [163]:
wta_con = wta_clustered.constituents(0.3)
wta_con.show()

[[[{pt: 1.41, eta: -1.08, phi: 2.71, E: 2.31}, {pt: 0.89, ...}], [...]],
 [[{pt: 1.88, eta: 0.576, phi: 0.406, E: 2.21}, ..., {pt: 110, eta: ..., ...}]],
 [[{pt: 0.648, eta: -0.224, phi: 0.252, E: 0.664}, {...}, ..., {pt: 110, ...}]],
 [[{pt: 0.341, eta: 1.6, phi: 3.07, E: 0.875}, ..., {pt: 0.497, ...}], ...],
 [[{pt: 0.726, eta: 0.154, phi: 0.347, E: 0.735}, ..., {pt: 0.369, ...}], ...],
 [[{pt: 0.355, eta: -1.61, phi: 2.07, E: 0.923}], ..., [{pt: 0.811, ...}, ...]],
 [[{pt: 0.512, eta: 1.14, phi: 5.44, E: 0.884}, ..., {pt: 1.82, ...}], ...],
 [[{pt: 0.665, eta: -1.32, phi: 3.18, E: 1.34}], [{pt: 0.69, ...}, ...]],
 [[{pt: 1.66, eta: 0.263, phi: 0.662, E: 1.72}, ..., {pt: 36.3, ...}], ...],
 [[{pt: 1.13, eta: 1.41, phi: 6.24, E: 2.44}, ..., {pt: 3.14, ...}], ...],
 ...,
 [[{pt: 0.759, eta: 0.836, phi: 6.14, E: 1.04}], [{pt: 0.859, ...}, ...]],
 [[{pt: 0.457, eta: 0.311, phi: 0.368, E: 0.479}, {...}, ..., {pt: 49.1, ...}]],
 [[{pt: 4.68, eta: -0.155, phi: 1.12, E: 4.74}, {...}, ..., {p

In [ ]:
# --- 1. Find the true hardest jet for the Standard Axis ---

# Find the index of the highest pT jet in each environment
std_max_pt_index = ak.argmax(std_jets.pt, axis=1, keepdims=True)

# Extract that exact jet to serve as your axis
leading_std = std_jets[std_max_pt_index][:, 0] 

# Extract the constituents belonging to that exact jet
# (We apply the exact same index mask to the constituents array)
std_constituents = std_clustered.constituents()[std_max_pt_index][:, 0]


# --- 2. Find the true hardest jet for the WTA Axis ---

wta_max_pt_index = ak.argmax(wta_jets.pt, axis=1, keepdims=True)
leading_wta = wta_jets[wta_max_pt_index][:, 0]

# Notice: You can use the std_constituents for your WTA N_ch and math, 
# because the particles inside the jet are the same, just the axis is different!

leading_std.show()


[{px: -403, py: -117, pz: -378, E: 574},
 {px: 559, py: -72.1, pz: 5.35, E: 573},
 {px: 198, py: 323, pz: -125, E: 411},
 {px: -322, py: -58.7, pz: 157, E: 382},
 {px: 323, py: -14.3, pz: -212, E: 399},
 {px: -37.6, py: 375, pz: -427, E: 592},
 {px: 398, py: -17.6, pz: 538, E: 677},
 {px: -337, py: 263, pz: -1.12e+03, E: 1.2e+03},
 {px: 513, py: 24.2, pz: -82.4, E: 555},
 {px: 281, py: -295, pz: 725, E: 839},
 ...,
 {px: 509, py: 45.1, pz: 81.4, E: 538},
 {px: 513, py: 627, pz: -182, E: 848},
 {px: 187, py: 457, pz: -501, E: 714},
 {px: 280, py: 220, pz: -188, E: 416},
 {px: -146, py: -317, pz: 326, E: 487},
 {px: 196, py: -359, pz: -278, E: 502},
 {px: 47.2, py: 436, pz: 66.1, E: 464},
 {px: -340, py: 529, pz: 23.2, E: 654},
 {px: 267, py: -296, pz: -327, E: 522}]


In [170]:
print(type(std_constituents))
print(type(std_clustered))
print(type(std_jets))
print(type(leading_std))

<class 'vector.backends.awkward.MomentumArray4D'>
<class 'fastjet._pyjet.AwkwardClusterSequence'>
<class 'vector.backends.awkward.MomentumArray4D'>
<class 'vector.backends.awkward.MomentumArray4D'>


In [179]:
print(std_constituents.type)
print(std_clustered)
print(std_jets.type)
print(leading_std.type)

1305 * option[var * Momentum4D[pt: float32, eta: float32, phi: float32, E: float32]]
1305 * var * Momentum4D[px: float64, py: float64, pz: float64, E: float64]
1305 * ?Momentum4D[px: float64, py: float64, pz: float64, E: float64]


In [183]:
print("constituents")
std_constituents.show()

std_clustered
print("jets")
std_jets.show()



constituents
[[{pt: 1.41, eta: -1.08, phi: 2.71, E: 2.31}, ..., {pt: 16.2, eta: ..., ...}],
 [{pt: 1.88, eta: 0.576, phi: 0.406, E: 2.21}, ..., {pt: 110, eta: ..., ...}],
 [{pt: 0.648, eta: -0.224, phi: 0.252, E: 0.664}, {...}, ..., {pt: 110, ...}],
 [{pt: 1, eta: 0.036, phi: 3.97, E: 1.01}, ..., {pt: 17.6, eta: 0.186, ...}],
 [{pt: 1.3, eta: 0.088, phi: 5.85, E: 1.3}, ..., {pt: 36.2, eta: -0.627, ...}],
 [{pt: 1.58, eta: -0.726, phi: 0.926, E: 2.02}, ..., {pt: 34.4, eta: ..., ...}],
 [{pt: 1.19, eta: 0.507, phi: 0.392, E: 1.35}, ..., {pt: 50.4, eta: 1.02, ...}],
 [{pt: 0.665, eta: -1.32, phi: 3.18, E: 1.34}, ..., {pt: 33.7, eta: ..., ...}],
 [{pt: 1.66, eta: 0.263, phi: 0.662, E: 1.72}, ..., {pt: 24.4, eta: ..., ...}],
 [{pt: 1.13, eta: 1.41, phi: 6.24, E: 2.44}, ..., {pt: 12.1, eta: 1.28, ...}],
 ...,
 [{pt: 0.598, eta: -0.472, phi: 0.514, E: 0.666}, {...}, ..., {pt: 12.3, ...}],
 [{pt: 0.457, eta: 0.311, phi: 0.368, E: 0.479}, {...}, ..., {pt: 49.1, ...}],
 [{pt: 4.68, eta: -0.155, 

In [184]:
print("leading")
leading_std.show()

leading
[{px: -403, py: -117, pz: -378, E: 574},
 {px: 559, py: -72.1, pz: 5.35, E: 573},
 {px: 198, py: 323, pz: -125, E: 411},
 {px: -322, py: -58.7, pz: 157, E: 382},
 {px: 323, py: -14.3, pz: -212, E: 399},
 {px: -37.6, py: 375, pz: -427, E: 592},
 {px: 398, py: -17.6, pz: 538, E: 677},
 {px: -337, py: 263, pz: -1.12e+03, E: 1.2e+03},
 {px: 513, py: 24.2, pz: -82.4, E: 555},
 {px: 281, py: -295, pz: 725, E: 839},
 ...,
 {px: 509, py: 45.1, pz: 81.4, E: 538},
 {px: 513, py: 627, pz: -182, E: 848},
 {px: 187, py: 457, pz: -501, E: 714},
 {px: 280, py: 220, pz: -188, E: 416},
 {px: -146, py: -317, pz: 326, E: 487},
 {px: 196, py: -359, pz: -278, E: 502},
 {px: 47.2, py: 436, pz: 66.1, E: 464},
 {px: -340, py: 529, pz: 23.2, E: 654},
 {px: 267, py: -296, pz: -327, E: 522}]


In [196]:
test = leading_std - std_jets
print(test)
print(test.type)
print(test.pt)
print(leading_std.pt)
print(std_jets.pt)
print(std_constituents.pt)

print(len(leading_std))
print(len(std_jets))
print(len(std_constituents))


[[{x: 0, y: 0, z: 0, t: 0}], ..., [{x: 266, y: -296, z: -326, ...}, {...}]]
1305 * option[var * Momentum4D[x: float64, y: float64, z: float64, t: float64]]
[[0], [0], [0], [327, 327, 0], ..., [409, 409, 0], [438, 0], [0], [398, 0]]
[419, 563, 379, 327, 323, 377, 398, 428, ..., 494, 356, 349, 409, 439, 629, 398]
[[419], [563], [379], [0.341, ..., 327], ..., [0.404, 439], [629], [0.607, 398]]
[[1.41, 0.844, 0.572, 0.89, 1.89, 1.06, ..., 5.34, 28.3, 8.92, 8.73, 16.2], ...]
1305
1305
1305


In [200]:
for i in range(5):
    print(i)
    print(leading_std[i])
    print("len jets:", len(std_jets[i]))
    print(std_jets[i])
    print(std_constituents[i])

0
{px: -403, py: -117, pz: -378, E: 574}
len jets: 1
[{px: -403, py: -117, pz: -378, E: 574}]
[{pt: 1.41, eta: -1.08, phi: 2.71, E: 2.31}, ..., {pt: 16.2, eta: -0.612, ...}]
1
{px: 559, py: -72.1, pz: 5.35, E: 573}
len jets: 1
[{px: 559, py: -72.1, pz: 5.35, E: 573}]
[{pt: 1.88, eta: 0.576, phi: 0.406, E: 2.21}, ..., {pt: 110, eta: 0.00997, ...}]
2
{px: 198, py: 323, pz: -125, E: 411}
len jets: 1
[{px: 198, py: 323, pz: -125, E: 411}]
[{pt: 0.648, eta: -0.224, phi: 0.252, E: 0.664}, ..., {pt: 110, eta: ..., ...}]
3
{px: -322, py: -58.7, pz: 157, E: 382}
len jets: 3
[{px: -0.34, py: 0.0243, pz: 0.806, E: 0.875}, ..., {px: -322, py: -58.7, ...}]
[{pt: 1, eta: 0.036, phi: 3.97, E: 1.01}, ..., {pt: 17.6, eta: 0.186, ...}]
4
{px: 323, py: -14.3, pz: -212, E: 399}
len jets: 3
[{px: 0.439, py: -0.0354, pz: -0.836, E: 0.945}, ..., {px: 323, py: -14.3, ...}]
[{pt: 1.3, eta: 0.088, phi: 5.85, E: 1.3}, ..., {pt: 36.2, eta: -0.627, ...}]


In [203]:
for i in range(11):
    if len(std_jets[i])>1:
        print(i)
        print("len jets:", len(std_jets[i]))
        print(leading_std[i])
        print(std_jets[i][-1])
        print(std_constituents[i])

3
len jets: 3
{px: -322, py: -58.7, pz: 157, E: 382}
{px: -322, py: -58.7, pz: 157, E: 382}
[{pt: 1, eta: 0.036, phi: 3.97, E: 1.01}, ..., {pt: 17.6, eta: 0.186, ...}]
4
len jets: 3
{px: 323, py: -14.3, pz: -212, E: 399}
{px: 323, py: -14.3, pz: -212, E: 399}
[{pt: 1.3, eta: 0.088, phi: 5.85, E: 1.3}, ..., {pt: 36.2, eta: -0.627, ...}]
5
len jets: 2
{px: -37.6, py: 375, pz: -427, E: 592}
{px: -37.6, py: 375, pz: -427, E: 592}
[{pt: 1.58, eta: -0.726, phi: 0.926, E: 2.02}, ..., {pt: 34.4, eta: ..., ...}]
6
len jets: 2
{px: 398, py: -17.6, pz: 538, E: 677}
{px: 398, py: -17.6, pz: 538, E: 677}
[{pt: 1.19, eta: 0.507, phi: 0.392, E: 1.35}, ..., {pt: 50.4, eta: 1.02, ...}]
7
len jets: 2
{px: -337, py: 263, pz: -1.12e+03, E: 1.2e+03}
{px: -337, py: 263, pz: -1.12e+03, E: 1.2e+03}
[{pt: 0.665, eta: -1.32, phi: 3.18, E: 1.34}, ..., {pt: 33.7, eta: -1.71, ...}]
10
len jets: 2
{px: 275, py: 111, pz: 305, E: 439}
{px: 275, py: 111, pz: 305, E: 439}
[{pt: 0.377, eta: 1.35, phi: 0.983, E: 0.777}, 

In [209]:
for i in range(1304):
    if len(std_jets[i])>1:
        if leading_std[i].pt != std_jets[i][-1].pt:
            print(i)
            print("len jets:", len(std_jets[i]))
            print(leading_std[i])
            print(std_jets[i][-1])
            print(std_constituents[i])

In [218]:
loosened_std = std_jets[:, -1]
print(loosened_std)
diff = loosened_std.pt - leading_std.pt
print(diff)
print(diff[diff>0])

[{px: -403, py: -117, pz: -378, E: 574}, ..., {px: 267, py: -296, pz: ..., ...}]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..., 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[]


# old stuff:

In [6]:
#from gemini - cross check
# optimising looping through the ttree
def optimize_tree_io(tree):
    # Disable every branch to prevent loading unused data into RAM
    tree.SetBranchStatus("*", 0) # "*" means all branches, 0 means not processing 
    
    # Explicitly enable ONLY the branches your analysis reads
    active_branches = [
        "genJetPt", "genJetEta", 
        "genDau_pt", "genDau_eta", "genDau_phi", "genDau_chg"
    ]

    for branch in active_branches:
        tree.SetBranchStatus(branch, 1) # 1 means being processed, so these branches get switched on

In [9]:
#storing jet data in a dictionary instead of looping through the whole tree multiple times


# ROOT histogram binning parameters     (i think this is better kept outside the function? maybe not though.)
eta_bins, eta_min, eta_max = 41, -6.15, 6.15
phi_bins = 33
phi_min = -(math.pi/2.0) + (math.pi/32.0)
phi_max = (3*math.pi/2.0) + (math.pi/32.0)

def initialize_all_bins(analysis_bins):
    '''
    initialising 
        1. the dictionary for jet info
        2. as well as initialising the histograms
    input: 
        - analysis bins
    mutated:
        - histograms with appropriate bin name are initialised but not returned
    output:
        - bin_data, the dictionary of jet info. Keys are multiplicity bins, and each key has items:
            - wta_stats: number of signal pairs, number of jets, and total Nch for this bin
            - std_stats: same but std
            - wta_hists: signal, background, and EPD hists
            - std_hists: same but std
        
    '''

    bin_data = {} #dict of dicts. The keys are multiplicity bins
    
    
    for mult_bin in analysis_bins:
        # Create a unique string identifier for the bin
        b_name = f"{mult_bin[0]}_{mult_bin[1]}" #not keeping the if statement that checks whether bin is one or two numbers because it should be 2 numbers.
            
        # Initialize dictionaries for this specific bin
        bin_data[b_name] = {
            'wta_stats': {'num_jets': 0, 'num_pairs': 0, 'Nch': 0},
            'std_stats': {'num_jets': 0, 'num_pairs': 0, 'Nch': 0},
            'wta_hists': {},
            'std_hists': {}
        }
        
        # Initialize WTA Histograms with SetDirectory(0)
        hSig_wta = ROOT.TH2D(f"hSig_WTA_{b_name}", ";#Delta#eta*;#Delta#phi*", eta_bins, eta_min, eta_max, phi_bins, phi_min, phi_max)
        hBkg_wta = ROOT.TH2D(f"hBkg_WTA_{b_name}", ";#Delta#eta*;#Delta#phi*", eta_bins, eta_min, eta_max, phi_bins, phi_min, phi_max)
        hEPD_wta = ROOT.TH2D(f"hEPD_WTA_{b_name}", ";#eta*;#phi*", 150, 0, 10, 120, -4, 4)
        
        for hist in [hSig_wta, hBkg_wta, hEPD_wta]:
            hist.SetDirectory(0)    #detaches the histograms from root gDirectory
            
        bin_data[b_name]['wta_hists'] = {'sig': hSig_wta, 'bkg': hBkg_wta, 'epd': hEPD_wta}
        
        # Initialize STD Histograms with SetDirectory(0)
        hSig_std = ROOT.TH2D(f"hSig_STD_{b_name}", ";#Delta#eta*;#Delta#phi*", eta_bins, eta_min, eta_max, phi_bins, phi_min, phi_max)
        hBkg_std = ROOT.TH2D(f"hBkg_STD_{b_name}", ";#Delta#eta*;#Delta#phi*", eta_bins, eta_min, eta_max, phi_bins, phi_min, phi_max)
        hEPD_std = ROOT.TH2D(f"hEPD_STD_{b_name}", ";#eta*;#phi*", 150, 0, 10, 120, -4, 4)
        
        for hist in [hSig_std, hBkg_std, hEPD_std]:
            hist.SetDirectory(0)
            
        bin_data[b_name]['std_hists'] = {'sig': hSig_std, 'bkg': hBkg_std, 'epd': hEPD_std}
        
    return bin_data




In [8]:
# helper function to put jet data into the dictionary

def get_bin_key(jet_mult, analysis_bins):
    '''
    outputs the multiplicity bin key for a given multiplicity
    
    inputs:
        - jet multiplicity
        - the analysis bins
    output:
        - the bin_data key for this multiplicity 
    '''

    for mult_bin in analysis_bins:
        if jet_mult >= mult_bin[0] and jet_mult < mult_bin[1]:
            return f"{mult_bin[0]}_{mult_bin[1]}"
    
    return None # Jet falls outside all defined bins

In [16]:
#running the analysis - one loop of the ttreee instead of many now.

def runAnalysis_SinglePass(tree, analysis_bins):
    '''
    runs the whole analysis. 
    inputs:
        - tree, the tracktree from the relevant root file
        - analysis_bins, the multiplicity bins we are using for the analysis
    '''
    
    # 1. Optimize IO footprint
    optimize_tree_io(tree) #makes sure only relevant branches are active for processing
    
    # 2. Setup master dictionary mapping all bins
    bin_data = initialize_all_bins(analysis_bins)
    
    # 3. The Single Pass Loop
    for event in tree:
        for ijet in range(event.genJetPt.size()): #ijet is an integer index for each jet
            
            std_pt = event.genJetPt[ijet]
            std_eta = event.genJetEta[ijet]
            
            if eventPass(std_pt, std_eta):  # making lab frame jet cuts
                dau_pts = event.genDau_pt[ijet]
                dau_etas = event.genDau_eta[ijet]
                dau_phis = event.genDau_phi[ijet]
                dau_charges = event.genDau_chg[ijet]

                jet_constituents = make_pseudojets(dau_pts, dau_etas, dau_phis, dau_charges) #check that make_pseudojets is optimised.

                # WTA Clustering
                wta_clustered = fastjet.ClusterSequence(jet_constituents, wta_def)
                wta_jets_tup = fastjet.sorted_by_pt(wta_clustered.inclusive_jets()) # is the sorting necessary? does this eat into compute time?
                
                if len(wta_jets_tup) > 0: # is this if statement necessary? will it ever be equal to zero?

                    wta_in_jet_frame, wta_mult = jetToJetFrame(wta_jets_tup[0]) # check that jetToJetFrame is optimised
                    b_name_wta = get_bin_key(wta_mult, analysis_bins)
                    
                    if b_name_wta: # i.e. it falls in one of the multiplicity bins (should be true always), this might be redundant.
                        hSig = bin_data[b_name_wta]['wta_hists']['sig']
                        hEPD = bin_data[b_name_wta]['wta_hists']['epd']
                        buildSignal(wta_in_jet_frame, hSig, hEPD)   # check that buildSignal is optimised and that it works with the right names, etc.
                        
                        pairs = len(wta_in_jet_frame) * (len(wta_in_jet_frame) - 1) // 2 # number of wta signal pairs in this jet
                        bin_data[b_name_wta]['wta_stats']['num_pairs'] += pairs # total number of pairs in the bin. used for background.
                        bin_data[b_name_wta]['wta_stats']['num_jets'] += 1      # number of jets in the bin. Used for bin's avg Nch per jet.
                        bin_data[b_name_wta]['wta_stats']['Nch'] += wta_mult    # total Nch for all jets in this bin. Avg Nch per jet is this divided by num_jets.

                # STD Clustering
                std_clustered = fastjet.ClusterSequence(jet_constituents, std_def)
                std_jets_tup = fastjet.sorted_by_pt(std_clustered.inclusive_jets()) # again is the sorting necessary?
                
                if len(std_jets_tup) > 0: #redundant or not?

                    std_in_jet_frame, std_mult = jetToJetFrame(std_jets_tup[0]) #check optimised
                    b_name_std = get_bin_key(std_mult, analysis_bins)
                    
                    if b_name_std:
                        hSig = bin_data[b_name_std]['std_hists']['sig']
                        hEPD = bin_data[b_name_std]['std_hists']['epd']
                        buildSignal(std_in_jet_frame, hSig, hEPD)           # check that names are fine
                        
                        pairs = len(std_in_jet_frame) * (len(std_in_jet_frame) - 1) // 2
                        bin_data[b_name_std]['std_stats']['num_pairs'] += pairs
                        bin_data[b_name_std]['std_stats']['num_jets'] += 1
                        bin_data[b_name_std]['std_stats']['Nch'] += std_mult

                # Critical C++ Memory Cleanup for FastJet Objects
                del wta_clustered   # don't need these cluster sequences anymore
                del std_clustered   
                
                #maybe delete std_jets_tup, std_in_jet_frame, anything else?
                
                gc.collect() #what exactly des this remove?

    # 4. Post-Processing: Loop over the populated dictionaries to build backgrounds and yields
    for b_name, data in bin_data.items():
        wta_stats = data['wta_stats']
        std_stats = data['std_stats']
        
        # Only process bins that actually caught jets
        if wta_stats['num_jets'] > 0 and std_stats['num_jets'] > 0:
            
            # Extract references
            hEPD_wta, hBkg_wta = data['wta_hists']['epd'], data['wta_hists']['bkg']
            hEPD_std, hBkg_std = data['std_hists']['epd'], data['std_hists']['bkg']
            
            # Build Backgrounds
            buildBkg(hEPD_wta, hBkg_wta, wta_stats['num_pairs'])
            buildBkg(hEPD_std, hBkg_std, std_stats['num_pairs'])


            #make sure epd and bkg are referencing different histogram objects every time the loop happens            

            # --- Yield processing and saveOutput() would be called here for this specific bin ---
            # Extract S/B ratios using your existing B(0,0) normalization logic

old stuff:

In [ ]:
#modified from Xiao's code
#clarify what the criteria is - he had something about 200 that i have not included

jetEtaCut = 1.6 #from PRL paper page 2
jetPtCut = 550  #from prl paper. This is the STANDARD pt, not wta pt.

def eventPass(jetPt, jetEta):  
    '''
    criteria for jet selection
    
    inputs:
        - jetEta: the jet's pseudorapidity, a positive or negative float
        - jetPt: the jet's momentum in GeV, a positive float             !!! USING STANDARD PT FOR CRITERIA, maybe later can try out with wta pt selection

    output: true if passed, false otherwise
    '''

    if(abs(jetEta)>jetEtaCut): return False
    if(jetPt < jetPtCut): return False

    return True



particlePtCut = 0.3 #from prl paper page 2
particleEtaCut = 2.4 #from prl paper page 2

def particlePass(particlePt, particleEta, particleCharge):
    '''
    criteria for particle selection

    inputs: 
        - particlePt: positive float for particle momentum magnitude
        - particleEta: float (positive or negative) for particle pseudorapidity
        - particleCharge: the particle's charge, not sure about the variable type

    output: true if passed, false otherwise
    '''
    if(particlePt < particlePtCut): return False
    if(abs(particleEta) > particleEtaCut): return False
    if (particleCharge is None or particleCharge==0): return False #only accepting charged particles

    return True



In [12]:
#from prelim file

def make_pseudojets(pts, etas, phis, charges):
    '''
    converts particles of one jet to a list of pseudojets. particle quality cuts are built in.
    inputs:
        - pts: a vector of particles for a given jet
        - etas: the corresponding pseudorapidities for each particle in the jet
        - phis: the corresponding azimuthal angles for each particle in the jet
        - charges: the charges of each particle in the jet
        
    output:
        - pj_list: a list of pseudojets, each pseudojet represents a particle in the jet
    '''
    
    pj_list = []

    for i in range(len(pts)): #looping through each particle in the jet
        
        if particlePass(pts[i], etas[i], charges[i]): #particle selection criteria
            p = fastjet.PseudoJet() #initialising a pseudojet
            p.reset_PtYPhiM(pts[i], etas[i], phis[i], 0.0) #defining the i-th particle as a pseudojet - is mass=0 fine?
            pj_list.append(p) #adding the particle to the list of pseudojets
    
    return pj_list #list of pseudojets


In [ ]:
#transforming values to jet frame

jt_lower = 0.3 #analysis can be redone with jt_lower = 0.5 as well.

def particleToJetFrame(jet_px, jet_py, jet_pz, particle):
    """
    Transforms one constituent particle to the jet frame. eta_star quality cuts reflected in 'include' output.
    
    Inputs:
        - jet_px, jet_py, jet_pz: The lab frame cartesian momentum 3-vector for the WHOLE jet
        - particle: A lab frame pseudoJet CONSTITUENT of that jet

    output:
        - jt: jet frame transverse momentum of the particle
        - eta_star: jet frame pseudorapidity of the particle
        - phi_star: jet frame phi of the particle
        - include: true if particle meets eta_star and jt criteria, false otherwise
    """

    # 1. Convert to ROOT TVector3
    # Use px, py, pz to ensure we have the full 3D vector
    p_jet = ROOT.TVector3(jet_px, jet_py, jet_pz)                 #momentum 3-vector of the whole jet
    p_part = ROOT.TVector3(particle.px(), particle.py(), particle.pz()) #momentum 3-vector of the particle

    # 1. Calculate Eta star (Relative Eta), discard non-qualifiers
    theta = p_part.Angle(p_jet)     #dot product angle between particle and jet momenta, range 0 to pi
    #cases for theta
    if theta == 0 or theta == math.pi: 
        return 0, 0, 0, False #on axis, not valid eta_star
    else: 
        eta_star = -math.log(math.tan(theta / 2.0)) #pseudorapidity formula for acceptable theta
        if abs(eta_star) > 5:
            return 0, 0, 0, False #include is false because absolute eta_star greater than 5 not allowed. 

        #what about eta_star less than 0.86? that is the anti-kt jet bdry. do i need to exclude stuff outside that?
        #eta_star values lower than 0.86 won't really happen because that is outside the theta = 0.8 anti-kt boundary.
        # so eta star is effectively limited to [0.86, 5]

    # 2. Calculate jT (Relative pT)
    jt = p_part.Perp(p_jet) #magnitude of part of particle momentum that is perpendicular to jet momentum
    if jt < jt_lower or jt > 3: #keeping 0.3geV<jt<3GeV . 
        return 0, 0, 0, False   #not soft so not plotted
    
    # 4. Calculate Phi star (Relative Phi)
    unit_jet = p_jet.Unit()
    z_axis = ROOT.TVector3(0, 0, 1)
    
    # Vector purely transverse to the jet axis
    v_pt = p_part - (unit_jet * p_part.Dot(unit_jet)) #magnitude of this is jt
    
    # Define the reference plane (jet axis & beam line)
    phi_origin = unit_jet.Cross(unit_jet.Cross(z_axis)) #phi = 0 vector, taken from Xiao's code
    phi_star = v_pt.Angle(phi_origin)
    
    # Determine the sign of phi_star
    if (phi_origin.Cross(v_pt.Unit())).Dot(unit_jet) < 0:
        phi_star = -phi_star

    return jt, eta_star, phi_star, True #if we reached this point without returning, then include should be true


def jetToJetFrame(labFrame_pj):
    '''
    transforms all constituents of a full jet from lab frame to the jet frame

    input: lab_pj, a jet pseudojet with coordinates measured in lab frame. The jet should already be clustered according to the desired scheme.

    outputs: 
        - ple_pj_list, a list of valid particle pseudojets with coordinates measured in the jet frame. (excludes hard jt)
        - jet_mult: the number of particles that meet in and out of jet criteria (doesn't exclude hard jt)
        note: len(ple_pj_list) != jet_mult because jet_mult includes hard particles and failed eta_star values
    '''
    #Extracting px, py, pz of the whole jet in lab frame
    jet_px =  labFrame_pj.px()
    jet_py = labFrame_pj.py()
    jet_pz = labFrame_pj.pz()
    
    ple_pj_list = [] #initialising list of particle pseudojets

    #looping through constituents
    jet_constituents = labFrame_pj.constituents()
    jet_mult = 0 #initialising multiplicity count
    for particle in jet_constituents:
        jet_mult+=1 # a valid clustered particle, but jt and eta_star not necessarily valid

        #finding jet frame coordinates for the particle
        jetFrame_pt, jetFrame_eta, jetFrame_phi, include = particleToJetFrame(jet_px, jet_py, jet_pz, particle)

        if include == True: #particle meets eta_star and jt criteria
            p_in_jet = fastjet.PseudoJet() #initialising a pseudojet     
            p_in_jet.reset_PtYPhiM(jetFrame_pt, jetFrame_eta, jetFrame_phi, 0.0) #redefining the particle in the jet frame (last index is mass)
            ple_pj_list.append(p_in_jet) #add the newly defined particle pseudojet to the list

        #maybe delete particle to free up memory?

    return ple_pj_list, jet_mult


In [25]:
#6 way histogram fill
def sixWayFill(hist, eta, phi, weight):
    '''
    helper function to do the 6-way symmetric histogram filling
    input: the 2d root histogram, the 2 inputs eta and phi, and a weight
    output: none, we are modifying the histogram
    '''
    hist.Fill(eta, phi, weight) #quadrant 1
    hist.Fill(-eta, phi, weight) #quadrant 2
    hist.Fill(-eta, -phi, weight) #quadrant 3
    hist.Fill(eta, -phi, weight) #quadrant 4
    hist.Fill(eta, 2*math.pi - phi, weight) #quadrant 1, phi wrap around pi
    hist.Fill(-eta, 2*math.pi - phi, weight) #quadrant 2, phi wrap around pi

In [ ]:
# do the 2 particle correlation

def buildSignal(pj_list, hSig, hEPD):
    '''
    fills in the signal and EPD histograms based on a jet represented by pj_list
    
    input: 
        - pj_list, a list of particle pseudojets for one jet. these should be in the jet frame, and in the right multiplicity bin
        - hSig, the signal histogram
        - hEPD, the event particle distribution histogram
    
    output:
        nothing. This just fills in the histograms.
    '''
    
    N_trig = len(pj_list) #JUST FOR THE LOOP, NOT THE HARD JT,FAILED ETA* INCLUSIVE MULTIPLICITY. that is jet_mult
    if N_trig < 2:
        return #no pairs can be made

    #build EPD
    for p in pj_list:
        corrected_phi = ROOT.TVector2.Phi_mpi_pi(p.phi()) #turning fastjet phi (0 to 2pi) into a -pi to pi range
        hEPD.Fill(p.eta(), corrected_phi, 1/N_trig) #filling in the trigger particle position in the EPD 


    #making the signal - is there a more efficient version of this than a nested for loop?
    for i in range(N_trig - 1):
        trig_eta = pj_list[i].eta()
        trig_phi = pj_list[i].phi()

        for j in range(i+1, N_trig): 
            track_eta = pj_list[j].eta()
            track_phi = pj_list[j].phi()
            delta_eta_star = abs(trig_eta - track_eta) #absolute dEta. Due to jet frame cuts, this is limited to 
            delta_phi_star = math.acos(math.cos(trig_phi - track_phi)) #limits dPhi to [0,pi]

            #weight = 1/Ntrig here, but it would involve the normalised energy product for an EEC
            #filling the signal symmetrically 6 ways
            sixWayFill(hSig, delta_eta_star, delta_phi_star, 1/N_trig) #weighting by 1/number of trigger particles in this jet
    
    #contribution of this one jet to signal and EPD are now filled
    #particles and pairs are weighted per trigger in the jet


In [ ]:
#build the background distribution once signal and EPD are filled
#method used here is the one described in the prl paper, not the one from github
def buildBkg(hEPD, hBkg_pre_corr, num_pairs):
    '''
    fills in the background histogram using the given EPD
    inputs:
        - hEPD, the signle-particle distribution
        - hBkg_pre_corr, the background histogram that needs filling, before it has been normalised by B(0,0)
        - num_pairs, the integer number of signal pairs.
    output:
        - fills and scales the background histogram so that it becomes hBkg, the corrected background histogram
    '''

    # 1. initialising c type coordinates
    eta1 = ctypes.c_double(0.0)
    phi1 = ctypes.c_double(0.0)
    eta2 = ctypes.c_double(0.0)
    phi2 = ctypes.c_double(0.0)

    # 2. loop to build background
    for _ in range(10*num_pairs): #background pairs are 10x the number of signal pairs, make sure it's an integer and not a float
        # Draw coords of two random particles from the EPD
        hEPD.GetRandom2(eta1, phi1)
        hEPD.GetRandom2(eta2, phi2)
        
        # 3. Extract the Python floats using .value
        e1 = eta1.value
        p1 = phi1.value
        e2 = eta2.value
        p2 = phi2.value
        
        # 4. Calculate kinematics using the extracted values
        delta_eta_star = abs(e1 - e2)
        delta_phi_star = math.acos(math.cos(p1 - p2))
        
        # 6-way fill for hBkg_pre_corr
        sixWayFill(hBkg_pre_corr, delta_eta_star, delta_phi_star, 1.0)

    #background is now filled the way described in the prl paper for the entire multiplicity bin
    #not yet corrected by B(0,0 though)

In [31]:
# NOT BEING USED RN
# Doing signal / background to get yield. 
def makeYield(hSig, hBkg, num_jets):
    '''
    makes yield. run once for wta, once for std
    '''
    #find B(0,0)
    origin_bin = hBkg.FindBin(0.0, 0.0)     # 1. Find the global bin index that corresponds to x=0.0, y=0.0
    B_00 = hBkg.GetBinContent(origin_bin)   # 2. Extract the number from that bin (this is B(0,0) , unscaled by ntrig)

    #find yield
    hYield = hSig.Clone("hYield_wta")
    hYield.Divide(hBkg)      # This does S(dEta, dPhi) / B(dEta, dPhi)
    hYield.Scale(B_00)       # This multiplies every bin by the single number B(0,0) (unscaled, but hBkg is also unscaled so it cancels)
    hYield.Scale(1/num_jets)    #averaging over all jets

    return hYield



In [ ]:
# --- SAVING THE OUTPUT ---

def saveOutput(bin_name, hYield_wta, hSig_wta, hBkg_wta, hEPD_wta, Nassoc_wta, avg_Nch_wta, hYield_std, hSig_std, hBkg_std, hEPD_std, Nassoc_std, avg_Nch_std):
    '''
    saves the 2d histograms
    '''
    # 1. Define your specific output folder path
    #output_dir = "/Users/rohanjagadeesan/Desktop/Code/li_lab/WinnerTakeAll/output/2pc_output_histograms"
    output_dir = "/home/rj65/winner_take_all/output"    # for nots

    # Create the folder if it doesn't already exist
    os.makedirs(output_dir, exist_ok=True)

    # file name
    filename = f"Yield_Histograms_Mult_{bin_name}.root"

    # 3. Combine the folder path and the filename
    full_filepath = os.path.join(output_dir, filename)

    # 4. Open the ROOT file using the FULL path
    out_file = ROOT.TFile(full_filepath, "RECREATE")

    # 5. write the histograms - naming already done in initialisation
    hYield_wta.Write()
    hYield_std.Write()
    hSig_wta.Write()
    hSig_std.Write()
    hBkg_wta.Write()
    hBkg_std.Write()
    hEPD_wta.Write()
    hEPD_std.Write()

    # 6. Create and Write the TParameters for Nassoc, and avg Nch per jet
    param_nassoc_wta = ROOT.TParameter('double')("Nassoc_WTA", Nassoc_wta)
    param_nassoc_std = ROOT.TParameter('double')("Nassoc_STD", Nassoc_std)
    param_nassoc_wta.Write()
    param_nassoc_std.Write()

    param_avg_Nch_wta = ROOT.TParameter('double')("avg_Nch_WTA", avg_Nch_wta)
    param_avg_Nch_std = ROOT.TParameter('double')("avg_Nch_STD", avg_Nch_std)
    param_avg_Nch_wta.Write()
    param_avg_Nch_std.Write()

    out_file.Close()

    print(f"Successfully saved yields to {full_filepath}")

### Cell below runs all the analysis, and saves the output

In [ ]:
#doing the thing for all the bins

analysis_bins = [ [0,25], [25,36], [36,48], [48,60], [60,71], [71,78], [78,91], [91,97], [97,1000] ]


for mult_bin in analysis_bins:
    runAnalysis(tree, mult_bin)

ROOT.gDirectory.Clear()

Successfully saved yields to /Users/rohanjagadeesan/Desktop/Code/li_lab/WinnerTakeAll/output/2pc_output_histograms/Yield_Histograms_Mult_0_25.root
Successfully saved yields to /Users/rohanjagadeesan/Desktop/Code/li_lab/WinnerTakeAll/output/2pc_output_histograms/Yield_Histograms_Mult_25_36.root
Successfully saved yields to /Users/rohanjagadeesan/Desktop/Code/li_lab/WinnerTakeAll/output/2pc_output_histograms/Yield_Histograms_Mult_36_48.root
Successfully saved yields to /Users/rohanjagadeesan/Desktop/Code/li_lab/WinnerTakeAll/output/2pc_output_histograms/Yield_Histograms_Mult_48_60.root
Successfully saved yields to /Users/rohanjagadeesan/Desktop/Code/li_lab/WinnerTakeAll/output/2pc_output_histograms/Yield_Histograms_Mult_60_71.root
Successfully saved yields to /Users/rohanjagadeesan/Desktop/Code/li_lab/WinnerTakeAll/output/2pc_output_histograms/Yield_Histograms_Mult_71_78.root
Successfully saved yields to /Users/rohanjagadeesan/Desktop/Code/li_lab/WinnerTakeAll/output/2pc_output_histogram